# 📊 NLP22 – n22dccn077 · Project Analysis Notebook
**English Writing Autocomplete – Comprehensive Dataset & Model Analysis**

| | |
|---|---|
| Nhóm | Nguyễn Duy Thái · Trần Nguyễn Sơn Thành · Cao Duy Thái |
| GV | ThS. Nguyễn Thị Tuyết Hải |
| Môn | NLP22 |

---

## Nội dung notebook
1. **Setup & Imports**
2. **Dataset Analysis** – Nguồn gốc, chất lượng, EDA chi tiết
3. **Data Pipeline** – Cách merge, filter, chia split
4. **train_small vs train_full** – Phân tích khi nào cần tập lớn hơn
5. **Model Evaluation** – Load checkpoints, tính đủ metrics
6. **Comparative Visualization** – Biểu đồ so sánh đầy đủ
7. **Inference Speed Analysis** – Latency thực tế
8. **Conclusions & Recommendations**

---
## 1. Setup & Imports

In [ ]:
import sys, os, math, time, json, re, pickle
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.ticker import LogFormatter
import seaborn as sns

# Optional: plotly cho interactive charts
try:
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    HAS_PLOTLY = True
    print('✓ Plotly available')
except ImportError:
    HAS_PLOTLY = False
    print('⚠ Plotly not installed – dùng matplotlib fallback')

# Style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#F8F9FA',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.dpi': 130,
})

PALETTE = {
    'ngram':      '#5C6BC0',
    'ngram_interp':'#7986CB',
    'hmm':        '#F57C00',
    'maxent':     '#E64A19',
    'lstm_std':   '#8E24AA',
    'lstm_beam':  '#AB47BC',
    'lstm_awd':   '#CE93D8',
    'gpt2':       '#00838F',
    'mamba':      '#2E7D32',
    'rag':        '#1565C0',
    'ensemble':   '#37474F',
}

# ── Project root ─────────────────────────────────────────────────
# Điều chỉnh ROOT cho phù hợp với máy bạn / Colab
ROOT = Path('.')   # chạy từ thư mục gốc NLP22-n22dccn077/
DATA_RAW  = ROOT / 'data' / 'raw'
DATA_PROC = ROOT / 'data' / 'processed'
CKPT_DIR  = ROOT / 'checkpoints'
LOG_DIR   = ROOT / 'logs'
LOG_DIR.mkdir(exist_ok=True)

print('✓ Setup complete')
print(f'  ROOT: {ROOT.resolve()}')
print(f'  data/processed: {DATA_PROC.resolve()}')
print(f'  checkpoints: {CKPT_DIR.resolve()}')

---
## 2. Dataset Analysis – Nguồn gốc và Chất lượng

### 2.1 Ba nguồn dữ liệu

In [ ]:
# ── Thông tin 3 nguồn dữ liệu ────────────────────────────────────
dataset_info = {
    'WikiText-103': {
        'source': 'Wikipedia Featured & Good Articles',
        'hf_id': 'wikitext / wikitext-103-raw-v1',
        'raw_size_mb': 536.97,
        'domain': 'Encyclopedic / Formal',
        'tokens_M': 103,
        'quality': 'Rất cao – biên tập bởi Wikipedia editors',
        'why': 'Chỉ lấy bài Featured/Good – pass review nghiêm ngặt',
        'pros': 'Ngữ pháp chuẩn, đa chủ đề, chuẩn benchmark quốc tế',
        'cons': 'Chỉ formal register, không có hội thoại'
    },
    'BBC News': {
        'source': 'BBC articles – tech/science/business/sport',
        'hf_id': 'SetFit/bbc-news',
        'raw_size_mb': 2.81,
        'domain': 'Journalism / News',
        'tokens_M': 0.5,
        'quality': 'Cao – editorial standards của BBC',
        'why': 'Cần văn phong báo chí để cover news register',
        'pros': 'Súc tích, rõ ràng, formal journalism style',
        'cons': 'Nhỏ (2.8MB), British English'
    },
    'arXiv Abstracts': {
        'source': 'Scientific paper abstracts',
        'hf_id': 'ccdv/arxiv-classification (no_ref)',
        'raw_size_mb': 57.21,
        'domain': 'Academic / Scientific',
        'tokens_M': 10,
        'quality': 'Rất cao – peer-reviewed academic writing',
        'why': 'Cần từ vựng học thuật cho domain academic writing',
        'pros': 'Dense academic vocab, từ khóa kỹ thuật phong phú',
        'cons': 'Quá formal, specialised vocabulary'
    }
}

# In bảng tổng hợp
print('='*90)
print(f'{"Dataset":<20} {"HuggingFace ID":<35} {"Size (MB)":<12} {"Domain"}')
print('='*90)
for name, info in dataset_info.items():
    print(f'{name:<20} {info["hf_id"]:<35} {info["raw_size_mb"]:<12} {info["domain"]}')
print('='*90)
print(f'TOTAL RAW: {sum(v["raw_size_mb"] for v in dataset_info.values()):.2f} MB')

print('\n📌 TẠI SAO CHỌN 3 NGUỒN NÀY thay vì Common Crawl hay Reddit?')
print('  → Common Crawl: 1000× lớn hơn nhưng ~30% câu sai ngữ pháp')
print('  → Reddit:       informal, slang, abbreviation nhiều')
print('  → Writing-assist tool: CẦN học từ văn bản đúng, không học từ văn bản lỗi!')
print('  → 3 nguồn này cover 3 register: Encyclopedic + News + Academic')

In [ ]:
# ── Visualize dataset composition ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Dataset Composition – NLP22 Autocomplete Project', fontsize=14, fontweight='bold')

# 1. Size pie chart
sizes = [d['raw_size_mb'] for d in dataset_info.values()]
labels = list(dataset_info.keys())
colors_pie = ['#5C6BC0', '#00838F', '#F57C00']
axes[0].pie(sizes, labels=labels, colors=colors_pie,
            autopct=lambda p: f'{p:.1f}%\n({p*597/100:.0f} MB)',
            startangle=90, textprops={'fontsize': 9})
axes[0].set_title('Raw Size Distribution (MB)')

# 2. Token count comparison
tokens = [d['tokens_M'] for d in dataset_info.values()]
bars = axes[1].bar(labels, tokens, color=colors_pie, edgecolor='white', linewidth=0.5)
axes[1].set_title('Approximate Token Count (Millions)')
axes[1].set_ylabel('Tokens (M)')
for bar, val in zip(bars, tokens):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val}M', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].tick_params(axis='x', labelsize=8)

# 3. Quality radar (subjective)
quality_scores = {
    'WikiText-103': {'Grammar': 9.5, 'Diversity': 9.0, 'Formal': 9.5, 'Size': 9.0, 'Clean': 9.5},
    'BBC News':     {'Grammar': 9.0, 'Diversity': 7.0, 'Formal': 8.5, 'Size': 3.0, 'Clean': 9.0},
    'arXiv':        {'Grammar': 9.5, 'Diversity': 6.0, 'Formal': 10.0, 'Size': 7.0, 'Clean': 9.5},
}
categories = list(list(quality_scores.values())[0].keys())
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]
ax_radar = fig.add_subplot(133, polar=True)
ax_radar.set_facecolor('#F8F9FA')
for (ds_name, scores), color in zip(quality_scores.items(), colors_pie):
    vals = list(scores.values()) + [list(scores.values())[0]]
    ax_radar.plot(angles, vals, color=color, linewidth=2, label=ds_name[:10])
    ax_radar.fill(angles, vals, color=color, alpha=0.15)
ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(categories, size=8)
ax_radar.set_ylim(0, 10)
ax_radar.set_title('Quality Dimensions\n(1-10, subjective)', pad=10)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=7)
# Hide the axes[2] which was replaced by polar
axes[2].set_visible(False)

plt.tight_layout()
plt.savefig('logs/fig_dataset_composition.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: logs/fig_dataset_composition.png')

### 2.2 Download Dataset (nếu chưa có)

In [ ]:
# ── Cách tải từng dataset – CHỈ CHẠY NẾU data/raw/ CHƯA CÓ ────────
# Nếu đã có đủ 3 file: bỏ qua cell này

RUN_DOWNLOAD = False  # ← đổi thành True nếu cần tải lại

if RUN_DOWNLOAD:
    from datasets import load_dataset
    DATA_RAW.mkdir(parents=True, exist_ok=True)

    # 1. WikiText-103 (HuggingFace official)
    print('Downloading WikiText-103...')
    wt = load_dataset('wikitext', 'wikitext-103-raw-v1', split='train')
    with open(DATA_RAW / 'wikitext.txt', 'w', encoding='utf-8') as f:
        for row in wt:
            text = row['text'].strip()
            if text:
                f.write(text + '\n')
    print(f'✓ WikiText saved: {(DATA_RAW/"wikitext.txt").stat().st_size/1e6:.1f} MB')

    # 2. BBC News (SetFit/bbc-news)
    print('Downloading BBC News...')
    bbc = load_dataset('SetFit/bbc-news', split='train')
    with open(DATA_RAW / 'bbc.txt', 'w', encoding='utf-8') as f:
        for row in bbc:
            text = row.get('text', row.get('article', '')).strip()
            if text:
                f.write(text + '\n')
    print(f'✓ BBC saved: {(DATA_RAW/"bbc.txt").stat().st_size/1e6:.1f} MB')

    # 3. arXiv Abstracts (ccdv/arxiv-classification)
    print('Downloading arXiv abstracts...')
    arxiv = load_dataset('ccdv/arxiv-classification', 'no_ref', split='train')
    with open(DATA_RAW / 'arxiv.txt', 'w', encoding='utf-8') as f:
        for row in arxiv:
            text = row.get('text', '').strip()[:2000]  # giới hạn 2000 ký tự/bài
            if text:
                f.write(text + '\n')
    print(f'✓ arXiv saved: {(DATA_RAW/"arxiv.txt").stat().st_size/1e6:.1f} MB')
else:
    # Verify files exist
    for fname in ['wikitext.txt', 'bbc.txt', 'arxiv.txt']:
        fpath = DATA_RAW / fname
        if fpath.exists():
            size = fpath.stat().st_size / 1e6
            print(f'✓ {fname}: {size:.2f} MB (already exists)')
        else:
            print(f'✗ {fname}: NOT FOUND – set RUN_DOWNLOAD=True to download')

---
## 3. Data Pipeline – Merge, Filter và Chia Split

In [ ]:
# ── Phân tích processed files ───────────────────────────────────
processed_files = {
    'train.txt':       {'split': 'train_full',  'pct': 90.0},
    'train_small.txt': {'split': 'train_small', 'pct': None},
    'val.txt':         {'split': 'val',         'pct': 5.0},
    'val_small.txt':   {'split': 'val_small',   'pct': None},
    'test.txt':        {'split': 'test',        'pct': 5.0},
    'test_small.txt':  {'split': 'test_small',  'pct': None},
}

print('=== Processed Files ===\n')
stats = {}
total_raw_mb = 596.99
for fname, meta in processed_files.items():
    fpath = DATA_PROC / fname
    if fpath.exists():
        size_mb = fpath.stat().st_size / 1e6
        # Count lines (sentences)
        with fpath.open() as f:
            n_lines = sum(1 for l in f if l.strip())
        stats[fname] = {'size_mb': size_mb, 'n_sents': n_lines, **meta}
        pct_str = f'{meta["pct"]}%' if meta['pct'] else 'head slice'
        print(f'  {fname:<22} {size_mb:>8.2f} MB  {n_lines:>9,} sents  [{pct_str}]')
    else:
        print(f'  {fname:<22}  NOT FOUND')
        stats[fname] = {'size_mb': 0, 'n_sents': 0, **meta}

print()
# Use known values if files not found
if stats.get('train.txt', {}).get('size_mb', 0) == 0:
    print('📌 Files not found locally – using known values from project')
    KNOWN_SIZES = {
        'train.txt': (402.88, 3_521_678),
        'train_small.txt': (27.47, 240_122),
        'val.txt': (22.42, 195_979),
        'val_small.txt': (2.75, 24_048),
        'test.txt': (22.33, 195_192),
        'test_small.txt': (2.76, 24_135),
    }
    for fname, (mb, sents) in KNOWN_SIZES.items():
        if fname in stats:
            stats[fname]['size_mb'] = mb
            stats[fname]['n_sents'] = sents

# Key insight
train_s = stats.get('train_small.txt', {}).get('size_mb', 27.47)
test_s  = stats.get('test.txt', {}).get('size_mb', 22.33)
print(f'\n⚠ train_small / test ratio: {train_s/test_s:.2f}x  (cần giải thích!)')
print('→ train_small có 240k câu, test có 195k câu → khác nhau hoàn toàn (shuffled trước split)')
print('→ KHÔNG có data leakage!')

In [ ]:
# ── Giải thích cơ chế preprocess.py ────────────────────────────
print('=== PREPROCESS.PY – LUỒNG XỬ LÝ ===\n')
print('Bước 1: Đọc TẤT CẢ file trong data/raw/*.txt')
print('        raw_files = sorted(RAW_DIR.glob("*.txt"))')
print('        → đọc: arxiv.txt, bbc.txt, wikitext.txt  (thứ tự alphabetical)\n')

print('Bước 2: Tách câu từng file, filter câu kém chất lượng')
print('        Bộ lọc áp dụng:')
filters = [
    ('Độ dài', '6 ≤ words ≤ 50', 'câu quá ngắn/dài bị bỏ'),
    ('Tỷ lệ chữ', '≥ 70% letter chars', 'lọc bảng biểu, code, số nhiều'),
    ('Unique words', '≥ 50% unique/total', 'lọc văn bản lặp lại'),
    ('Dấu câu', 'kết thúc bằng . ! ?', 'lọc câu bị cắt giữa chừng'),
    ('Wiki headings', 'regex ^\s*=.*=\s*$', 'lọc = = History = = v.v.'),
    ('URL', 'regex https?://\S+', 'xóa đường dẫn web'),
]
for f_name, f_rule, f_why in filters:
    print(f'        [{f_name:<15}] {f_rule:<35} → {f_why}')
print()

print('Bước 3: Lowercase + NFKC normalize')
print('        → giảm vocab size, tránh bias với từ đầu câu viết hoa\n')

print('Bước 4: Shuffle ALL sentences (seed=42) → trộn 3 nguồn')
print('        rng = random.Random(42)')
print('        rng.shuffle(all_sents)  # 3 nguồn được trộn ngẫu nhiên\n')

print('Bước 5: Chia 90/5/5 → train.txt / val.txt / test.txt')
print()
print('Bước 6 (riêng biệt – KHÔNG phải preprocess.py):')
print('        train_small.txt = head 6.8% của train.txt  (đơn giản như: head -n N)')
print('        val_small.txt   = head 12.3% của val.txt')
print('        test_small.txt  = head 12.4% của test.txt\n')

print('=== KẾT QUẢ: Tại sao raw (597MB) > processed (448MB)? ===')
filtered = 596.99 - (402.88 + 22.42 + 22.33)
print(f'Raw total:         596.99 MB')
print(f'Processed total:   447.63 MB')
print(f'Filtered out:     {filtered:.2f} MB = {filtered/596.99*100:.1f}% bị loại bỏ')
print('Lý do chính: WikiText có NHIỀU tiêu đề (= =..= =), bảng, câu ngắn bị lọc')

In [ ]:
# ── Visualize: Data pipeline flow ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 1. Sankey-style flow (bar chart)
stages = ['Raw Total', 'After Filter\n(−25%)', 'train.txt\n(90%)', 'val.txt\n(5%)', 'test.txt\n(5%)']
values = [596.99, 447.63, 402.88, 22.42, 22.33]
bar_colors = ['#5C6BC0', '#66BB6A', '#26A69A', '#FFA726', '#EF5350']
bars = axes[0].bar(stages, values, color=bar_colors, edgecolor='white', linewidth=0.8, width=0.6)
axes[0].set_title('Data Pipeline: Raw → Processed (MB)', fontweight='bold')
axes[0].set_ylabel('Size (MB)')
for bar, val in zip(bars, values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                f'{val:.0f} MB', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].axhline(y=447.63, color='gray', linestyle='--', alpha=0.5, label='Post-filter level')
axes[0].legend(fontsize=9)

# 2. Split proportions (stacked horizontal)
splits = {'train_full': 402.88, 'val': 22.42, 'test': 22.33}
total = sum(splits.values())
split_colors = ['#26A69A', '#FFA726', '#EF5350']
lefts = 0
for (name, val), color in zip(splits.items(), split_colors):
    pct = val / total * 100
    axes[1].barh(['Processed splits'], val, left=lefts, color=color, label=f'{name}: {val:.0f}MB ({pct:.1f}%)')
    axes[1].text(lefts + val/2, 0, f'{pct:.0f}%', ha='center', va='center',
                fontsize=10, fontweight='bold', color='white')
    lefts += val

# Add train_small annotation
axes[1].barh(['train_small breakdown'], 27.47, color='#26A69A', alpha=0.7, label='train_small: 27MB (6.8%)')
axes[1].barh(['train_small breakdown'], 402.88-27.47, left=27.47, color='#B2DFDB', alpha=0.7, label='rest of train')
axes[1].text(13.7, 1, '6.8%', ha='center', va='center', fontsize=10, fontweight='bold', color='white')
axes[1].text(215, 1, '93.2%', ha='center', va='center', fontsize=10, fontweight='bold', color='#37474F')

axes[1].set_title('Train/Val/Test Split và train_small', fontweight='bold')
axes[1].set_xlabel('Size (MB)')
axes[1].legend(loc='lower right', fontsize=8, ncol=2)
axes[1].set_xlim(0, 480)

plt.tight_layout()
plt.savefig('logs/fig_data_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. EDA – Exploratory Data Analysis chi tiết

In [ ]:
# ── EDA: Đọc mẫu từ processed files ────────────────────────────
def sample_file(path, n=5000, seed=42):
    """Đọc n dòng ngẫu nhiên từ file."""
    rng = np.random.RandomState(seed)
    lines = []
    if not path.exists():
        print(f'⚠ {path.name} not found')
        return []
    with path.open(encoding='utf-8') as f:
        all_lines = [l.strip() for l in f if l.strip()]
    idx = rng.choice(len(all_lines), min(n, len(all_lines)), replace=False)
    return [all_lines[i] for i in idx]

# Lấy mẫu từ train_small và test
print('Sampling files for EDA...')
sample_train = sample_file(DATA_PROC / 'train_small.txt', n=5000)
sample_test  = sample_file(DATA_PROC / 'test.txt',        n=5000)
sample_val   = sample_file(DATA_PROC / 'val.txt',         n=2000)

if not sample_train:
    # Fallback: generate mock data để minh họa
    print('📌 Dùng mock data vì file không tìm thấy')
    mock_sentences = [
        'the quick brown fox jumps over the lazy dog',
        'in this paper we propose a novel approach to language modeling',
        'the results demonstrate that our method outperforms existing baselines',
        'according to the report published by the government',
        'machine learning has revolutionized natural language processing',
    ] * 1000
    sample_train = mock_sentences
    sample_test  = mock_sentences[:1000]

# Tính thống kê
def compute_stats(sentences):
    lengths = [len(s.split()) for s in sentences]
    all_words = [w for s in sentences for w in s.split()]
    vocab = Counter(all_words)
    return {
        'n_sents': len(sentences),
        'mean_len': np.mean(lengths),
        'median_len': np.median(lengths),
        'std_len': np.std(lengths),
        'min_len': min(lengths),
        'max_len': max(lengths),
        'vocab_size': len(vocab),
        'total_tokens': len(all_words),
        'lengths': lengths,
        'vocab': vocab,
    }

stats_train = compute_stats(sample_train)
stats_test  = compute_stats(sample_test)

print(f'\n{"Metric":<22} {"train_small (sample)":>22} {"test (sample)":>18}')
print('-'*65)
for key in ['n_sents', 'mean_len', 'median_len', 'std_len', 'vocab_size', 'total_tokens']:
    vt = stats_train[key]
    vx = stats_test[key]
    print(f'{key:<22} {vt:>22.2f} {vx:>18.2f}')

In [ ]:
# ── EDA Visualizations ──────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('EDA – Processed Dataset Analysis', fontsize=14, fontweight='bold')

# 1. Sentence length distribution
ax = axes[0, 0]
ax.hist(stats_train['lengths'], bins=45, alpha=0.7, color='#5C6BC0', label='train_small', density=True)
ax.hist(stats_test['lengths'],  bins=45, alpha=0.5, color='#EF5350', label='test', density=True)
ax.axvline(stats_train['mean_len'], color='#5C6BC0', linestyle='--', linewidth=1.5,
           label=f'train mean={stats_train["mean_len"]:.1f}')
ax.axvline(stats_test['mean_len'],  color='#EF5350', linestyle='--', linewidth=1.5,
           label=f'test mean={stats_test["mean_len"]:.1f}')
ax.set_xlabel('Sentence Length (words)')
ax.set_ylabel('Density')
ax.set_title('Sentence Length Distribution')
ax.legend(fontsize=8)

# 2. Vocabulary frequency (Zipf law)
ax = axes[0, 1]
vocab_train = stats_train['vocab']
freqs = sorted(vocab_train.values(), reverse=True)[:1000]
ranks = list(range(1, len(freqs)+1))
ax.loglog(ranks, freqs, color='#5C6BC0', linewidth=1.5, alpha=0.8, label='Observed')
# Zipf line
zipf = [freqs[0] / r for r in ranks]
ax.loglog(ranks, zipf, 'r--', linewidth=1, alpha=0.6, label="Zipf's Law f∝1/r")
ax.set_xlabel('Rank')
ax.set_ylabel('Frequency')
ax.set_title("Word Frequency (Zipf's Law check)")
ax.legend(fontsize=8)

# 3. Top 20 most common words
ax = axes[0, 2]
top_words = [(w, c) for w, c in vocab_train.most_common(25)
             if w not in {'the', 'a', 'of', 'and', 'in', 'to', 'is', 'was', 'it', 'that', 'he',
                          'she', 'for', 'on', 'are', 'as', 'with', 'his', 'they', 'at', 'be',
                          'from', 'or', 'an', 'this'}][:15]
words_x = [w for w, c in top_words]
counts   = [c for w, c in top_words]
ax.barh(words_x[::-1], counts[::-1], color='#66BB6A', edgecolor='white')
ax.set_title('Top Content Words (excl. stopwords)')
ax.set_xlabel('Frequency in sample')

# 4. Sentence length box plot by source (simulated)
ax = axes[1, 0]
# Since we merged sources, simulate different length distributions
wiki_lens  = np.random.normal(22, 8, 3000).clip(6, 50).astype(int)
bbc_lens   = np.random.normal(18, 6, 500).clip(6, 50).astype(int)
arxiv_lens = np.random.normal(26, 7, 1500).clip(6, 50).astype(int)
ax.boxplot([wiki_lens, bbc_lens, arxiv_lens],
           labels=['WikiText\n(~83%)', 'BBC News\n(~8%)', 'arXiv\n(~9%)'],
           patch_artist=True,
           boxprops=dict(facecolor='#E3F2FD'),
           medianprops=dict(color='#1565C0', linewidth=2))
ax.set_ylabel('Sentence Length (words)')
ax.set_title('Length by Source (estimated)')
ax.axhline(y=20, color='gray', linestyle='--', alpha=0.5, label='Filter min (6) / common mean')

# 5. Token count comparison across splits
ax = axes[1, 1]
split_names = ['train.txt', 'train_small', 'val.txt', 'test.txt']
split_sents = [3_521_678, 240_122, 195_979, 195_192]
bar_colors2 = ['#26A69A', '#80CBC4', '#FFA726', '#EF5350']
bars2 = ax.bar(split_names, [s/1000 for s in split_sents],
               color=bar_colors2, edgecolor='white')
for bar, val in zip(bars2, split_sents):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
           f'{val/1000:.0f}k', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Sentence Count per Split (thousands)')
ax.set_ylabel('Sentences (k)')
ax.tick_params(axis='x', labelsize=8)

# 6. Filter impact
ax = axes[1, 2]
filter_names = ['Len < 6\nwords', 'Len > 50\nwords', 'Wiki\nheadings', 'Low letter\nratio', 'Low unique\nwords', 'No\npunctuation']
filter_impact = [3.5, 4.2, 8.1, 6.3, 2.1, 1.8]  # estimated % of total removed by each
bar_colors3 = ['#EF9A9A', '#F48FB1', '#CE93D8', '#90CAF9', '#80DEEA', '#A5D6A7']
ax.bar(filter_names, filter_impact, color=bar_colors3, edgecolor='white')
ax.set_title('Estimated % Removed by Each Filter')
ax.set_ylabel('% of raw sentences removed')
ax.tick_params(axis='x', labelsize=7)
total_removed = sum(filter_impact)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.text(2.5, max(filter_impact)*0.9, f'Total filtered: ~{total_removed:.0f}% (overlaps possible)',
        ha='center', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('logs/fig_eda.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. train_small vs train_full – Khi nào cần tập lớn hơn?

In [ ]:
# ── Phân tích train_small adequacy ──────────────────────────────
print('=== CÂU HỎI: train_small/test ≈ 1.23x – Có ổn không? ===\n')

print('📌 ĐIỂM QUAN TRỌNG NHẤT: inference speed ≠ training data size!')
print('   Training size → model QUALITY (PPL, accuracy) – chỉ ảnh hưởng lúc train')
print('   Architecture  → inference SPEED (ms/query) – khi demo cho giáo viên')
print()

print('✓ test.txt và train_small.txt KHÔNG có overlap:')
print('  preprocess.py shuffle ALL sentences (seed=42) TRƯỚC KHI chia split')
print('  → train_small = đầu 6.8% của train.txt (90% đầu của shuffled data)')
print('  → test.txt = 5% cuối của shuffled data')
print('  → Hai tập HOÀN TOÀN khác nhau → KHÔNG data leakage')
print()

# Model adequacy table
print('=== TRAIN_SMALL (240k câu) ĐỦ KHÔNG cho từng model? ===')
print()
model_adequacy = [
    ('N-gram KN-4',    'Đủ',      'Tốt', 'N-gram đếm frequencies – 240k câu đủ coverage'),
    ('N-gram Interp',  'Đủ',      'Tốt', 'Tương tự KN-4, learn lambda từ val'),
    ('HMM-LM',         'Đủ',      'OK',  'POS bigram đơn giản, ít tham số'),
    ('MaxEnt LM',      'Đủ',      'OK',  'Online SGD 3 epochs, 240k đủ'),
    ('LSTM Standard',  'Chấp nhận','OK', '240k là ít, nhưng demo được PPL~118'),
    ('AWD-LSTM',       'Chấp nhận','Tốt','Regularization mạnh → học tốt từ ít data'),
    ('ELMo+LSTM',      'Chấp nhận','OK', 'BiLM cần nhiều data hơn, nhưng chấp nhận được'),
    ('GPT-2 FT',       'Đã dùng full','Rất tốt','Đã fine-tune trên 3.5M câu ✓'),
    ('Mamba SSM',      'Chấp nhận','Tốt','SSM hiệu quả với ít data hơn Transformer'),
]
print(f'{"Model":<18} {"train_small":<16} {"Chất lượng":<12} {"Lý do"}')
print('-'*80)
for model, adequacy, quality, reason in model_adequacy:
    marker = '✓' if 'Đủ' in adequacy or 'full' in adequacy else '~'
    print(f'{marker} {model:<17} {adequacy:<16} {quality:<12} {reason}')

print()
print('=== CÓ NÊN TẠO train_small LỚN HƠN KHÔNG? ===')
print()
print('Khuyến nghị: TẠO train_medium.txt = 50MB (~450k câu) thay vì 27MB')
print('Lý do:')
print('  • LSTM/Mamba PPL có thể giảm thêm 8-12% (ước tính)')
print('  • Vocab coverage tốt hơn → ít <UNK> hơn')
print('  • Train time chỉ tăng ~2x (vẫn chấp nhận được: ~50 phút T4)')
print('  • test/val giữ nguyên → so sánh fair')
print()
print('Cách tạo train_medium.txt:')
print('  # Python')
print('  with open("data/processed/train.txt") as f:')
print('      lines = f.readlines()')
print('  with open("data/processed/train_medium.txt", "w") as f:')
print('      f.writelines(lines[:450000])  # ~50MB')

In [ ]:
# ── Inference Speed Analysis ─────────────────────────────────────
print('=== TỐC ĐỘ INFERENCE – KHÔNG LIÊN QUAN ĐẾN TRAINING DATA SIZE ===\n')

speed_data = {
    'N-gram KN-4':   {'ms': 0.5,   'group': 'Statistical', 'ppl': 1088},
    'N-gram Interp': {'ms': 1.0,   'group': 'Statistical', 'ppl': 900},
    'HMM-LM':        {'ms': 5.0,   'group': 'Statistical', 'ppl': 600},
    'MaxEnt LM':     {'ms': 4.0,   'group': 'Statistical', 'ppl': None},
    'LSTM Standard': {'ms': 10.0,  'group': 'Neural RNN',  'ppl': 118},
    'AWD-LSTM':      {'ms': 13.0,  'group': 'Neural RNN',  'ppl': 100},
    'ELMo+LSTM':     {'ms': 22.0,  'group': 'Neural RNN',  'ppl': 110},
    'Mamba SSM':     {'ms': 25.0,  'group': 'SSM 2023',    'ppl': 55},
    'GPT-2 FT':      {'ms': 65.0,  'group': 'Transformer', 'ppl': 42.74},
    'RAG Hybrid':    {'ms': 120.0, 'group': 'Retrieval',   'ppl': None},
    'RRF Ensemble':  {'ms': 130.0, 'group': 'Ensemble',    'ppl': None},
}

DEMO_THRESHOLD = 100  # ms – ngưỡng chấp nhận cho real-time autocomplete

print(f'Ngưỡng acceptable cho autocomplete: < {DEMO_THRESHOLD}ms\n')
print(f'{"Model":<18} {"Latency":>10} {"Status":<12} {"Group"}')
print('-'*60)
for model, d in speed_data.items():
    ok = '✓ Fast' if d['ms'] < DEMO_THRESHOLD else '⚠ Slow'
    print(f'{model:<18} {d["ms"]:>8.1f} ms  {ok:<12} {d["group"]}')

print()
print('📌 RAG và Ensemble chậm vì retrieval overhead, không phải model size')
print('   Khi demo: disable RAG/Ensemble nếu muốn responsive UI')
print('   GPT-2 (~65ms) vẫn ổn cho demo trực tiếp')

---
## 6. Model Evaluation – Load Checkpoints và Tính Metrics

# ── 📈 Model Evaluation Results Analysis ─────────────────

Trong phần này, chúng ta sẽ load kết quả đã được evaluate từ `logs/eval_results.json` và trực quan hóa các metrics quan trọng của các mô hình: Perplexity, Accuracy (Top-k), MRR, và Latency.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Load results
ROOT = Path.cwd().parent
json_path = ROOT / "logs" / "eval_results.json"

results = {}
if json_path.exists():
    with open(json_path, "r", encoding="utf-8") as f:
        results = json.load(f)
else:
    print("File eval_results.json không tồn tại. Vui lòng chạy `python -m src.evaluate_all` trước.")

# Lọc ra các model đã được train và có dữ liệu
trained_results = {k: v for k, v in results.items() if v.get('trained', False) and v.get('ppl') is not None}

df = pd.DataFrame.from_dict(trained_results, orient="index")
df = df.reset_index().rename(columns={"index": "Model"})

# Hiển thị bảng tổng hợp
display(df.style.background_gradient(cmap="viridis").format(precision=3))

### 📉 1. Perplexity (Độ đo ngôn ngữ) - Càng thấp càng tốt

In [ ]:
plt.figure(figsize=(10, 5))
df_ppl = df.dropna(subset=['ppl']).sort_values('ppl', ascending=False)
sns.barplot(data=df_ppl, x='ppl', y='Model', palette='coolwarm', hue='Model', legend=False)
plt.title("Perplexity của các mô hình (Thấp hơn là tốt hơn)")
plt.xlabel("Perplexity")
plt.ylabel("")
for i, v in enumerate(df_ppl['ppl']):
    plt.text(v + min(df_ppl['ppl'])*0.05, i, f"{v:.2f}", va='center')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

### 🎯 2. Top-k Accuracy (Độ chính xác Top-1 và Top-5) - Càng cao càng tốt

In [ ]:
df_acc = df.dropna(subset=['top1', 'top5']).sort_values('top5', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
df_acc.plot(x='Model', y=['top1', 'top5'], kind='barh', ax=ax, color=['#26C6DA', '#66BB6A'])
plt.title("Top-1 và Top-5 Accuracy của các mô hình")
plt.xlabel("Accuracy")
plt.ylabel("")
plt.legend(["Top-1", "Top-5"])
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.xlim(0, max(df_acc['top5']) * 1.15)
plt.show()

### 🔢 3. Mean Reciprocal Rank (MRR) - Càng cao càng tốt

In [ ]:
plt.figure(figsize=(10, 5))
df_mrr = df.dropna(subset=['mrr']).sort_values('mrr', ascending=False)
sns.barplot(data=df_mrr, x='mrr', y='Model', palette='magma', hue='Model', legend=False)
plt.title("Mean Reciprocal Rank (MRR) của các mô hình")
plt.xlabel("MRR")
plt.ylabel("")
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

### ⚡ 4. Latency (Độ trễ khi dự đoán) - Càng thấp càng tốt

In [ ]:
plt.figure(figsize=(10, 5))
df_lat = df.dropna(subset=['latency_ms']).sort_values('latency_ms', ascending=False)
sns.barplot(data=df_lat, x='latency_ms', y='Model', palette='YlOrRd', hue='Model', legend=False)
plt.title("Latency trung bình (ms/query) của các mô hình")
plt.xlabel("Latency (ms)")
plt.ylabel("")
for i, v in enumerate(df_lat['latency_ms']):
    plt.text(v + max(df_lat['latency_ms'])*0.02, i, f"{v:.1f} ms", va='center')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

### 🕷️ 5. Radar Chart Tổng Hợp

In [ ]:
import numpy as np

# Chọn các models để vẽ radar chart
radar_df = df.dropna(subset=['top1', 'top5', 'mrr']).copy()

if len(radar_df) > 0:
    categories = ['Top-1', 'Top-5', 'MRR']
    N = len(categories)

    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    plt.xticks(angles[:-1], categories, size=12)

    for i, row in radar_df.iterrows():
        values = [row['top1'], row['top5'], row['mrr']]
        values += values[:1]
        ax.plot(angles, values, linewidth=2, linestyle='solid', label=row['Model'])
        ax.fill(angles, values, alpha=0.1)

    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.title("So sánh tổng hợp (Radar Chart)", size=15, y=1.1)
    plt.show()
else:
    print("Chưa đủ dữ liệu để vẽ radar chart.")